In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub

## Load dependencies

In [ ]:
import evaluate, torch
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

## Config

In [ ]:
user_name = "amin-oj"
dataset_name = "imdb"
model_name = "distilbert/distilbert-base-uncased"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
ckpt_name = "imdb_text_classification_small"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
imdb = load_dataset(dataset_name)
imdb["train"][0]

There are two fields in this dataset:

- `text`: the movie review text.
- `label`: a value that is either `0` for a negative review or `1` for a positive review.

## Preprocess

In [ ]:
def preprocess_function(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_imdb = imdb.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    fn_kwargs = {"tokenizer": tokenizer}
)

In [ ]:
tokenized_imdb["train"][0].keys()

## Train

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation,
# instead of padding the whole dataset to the maximum length.

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    push_to_hub=push_to_hub,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none" # to disable wandb login
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"].shuffle().select(range(5000)),
    eval_dataset=tokenized_imdb["test"].shuffle().select(range(1000)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Inference

In [ ]:
text = """
This was a masterpiece. Not completely faithful to the books,
but enthralling from beginning to end. Might be my favorite of the three.
"""

### Simplest way

In [ ]:
classifier = pipeline("sentiment-analysis", model=f"{user_name}/{ckpt_name}")
classifier(text)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt")
model = AutoModelForSequenceClassification.from_pretrained(f"{user_name}/{ckpt_name}")
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]